In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import aera

# ============================================================
# Input / output
# ============================================================
SRC = Path(
    "/nird/datalake/NS2980K/users/yongyub/O2_linearlity/"
    "TipESM/cmor/esm-up2p0/v20251010/AERA/input/references/"
    "NorESM2-LM_esm-hist_r1i1p1f1_1850-2014_annual_global_timeseries.csv"
)

OUT_FULL = SRC.with_name(
    "NorESM2-LM_esm-hist_r1i1p1f1_1850-2014_AERA_full_input.csv"
)

OUT_MIN = SRC.with_name(
    "NorESM2-LM_esm-hist_r1i1p1f1_1850-2014_AERA_temp_ff_input.csv"
)

y0 = 1850
y_hist_end = 2014

# ============================================================
# Read historical global annual time series
# ============================================================
hist = pd.read_csv(SRC)
hist["year"] = hist["year"].astype(int)
hist = hist.set_index("year").sort_index()

target_years = np.arange(y0, y_hist_end + 1)

missing = target_years[~np.isin(target_years, hist.index.values)]
if len(missing) > 0:
    raise ValueError(
        f"Missing historical years in source CSV: {missing[:10]}"
    )

# ============================================================
# Minimal AERA input: only what we overwrite/use explicitly
# ============================================================
df_min = pd.DataFrame(index=target_years)
df_min.index.name = "year"

# Keep temperature as absolute K.
# Then future model TREFHT should also be kept as absolute K.
df_min["temp"] = hist.loc[target_years, "tas_global_mean_K"].astype(float)

# Use historical CO2 surface emission as fossil-fuel emission input.
df_min["ff_emission"] = hist.loc[
    target_years, "co2_surface_emission_PgC_yr"
].astype(float)

df_min.to_csv(OUT_MIN)

# ============================================================
# Full AERA dataframe input
# ============================================================
df = aera.get_base_df()

df.loc[target_years, "temp"] = df_min["temp"].values
df.loc[target_years, "ff_emission"] = df_min["ff_emission"].values

# Save full dataframe.
# This keeps default lu_emission and non_co2_emission from AERA.
df.to_csv(OUT_FULL)

print(f"Wrote minimal input: {OUT_MIN}")
print(f"Wrote full AERA input: {OUT_FULL}")

print("\nCheck:")
print(df.loc[1850:1855, ["temp", "ff_emission", "lu_emission", "non_co2_emission"]])
print(df.loc[2010:2014, ["temp", "ff_emission", "lu_emission", "non_co2_emission"]])

Use the following non-CO2 emission file: /nird/home/yongyub/kimyy/Dropbox/source/python/all/AERA/AERA_master_260702/AERA/aera/data/nonco2_emis_ssp126_v3.dat
Use the following land use emission file: /nird/home/yongyub/kimyy/Dropbox/source/python/all/AERA/AERA_master_260702/AERA/aera/data/lu_emis_ssp126_bern3d_adj_GCB2020_v1.dat
Use the following historical fossil fuel CO2 emission file: /nird/home/yongyub/kimyy/Dropbox/source/python/all/AERA/AERA_master_260702/AERA/aera/data/co2_ff_GCP_plus_NDC_v1.dat
Wrote minimal input: /nird/datalake/NS2980K/users/yongyub/O2_linearlity/TipESM/cmor/esm-up2p0/v20251010/AERA/input/references/NorESM2-LM_esm-hist_r1i1p1f1_1850-2014_AERA_temp_ff_input.csv
Wrote full AERA input: /nird/datalake/NS2980K/users/yongyub/O2_linearlity/TipESM/cmor/esm-up2p0/v20251010/AERA/input/references/NorESM2-LM_esm-hist_r1i1p1f1_1850-2014_AERA_full_input.csv

Check:
            temp  ff_emission  lu_emission  non_co2_emission
year                                             

In [2]:
import numpy as np
import pandas as pd
from pathlib import Path

# Use your installed AERA/OA package
# If you installed OA version as aera_oa, use:
# import aera_oa as aera
import aera_oa


# ============================================================
# Input / output
# ============================================================
SRC = Path(
    "/nird/datalake/NS2980K/users/yongyub/O2_linearlity/"
    "TipESM/cmor/esm-up2p0/v20251010/AERA/input/references/"
    "NorESM2-LM_esm-hist_r1i1p1f1_1850-2014_annual_global_timeseries.csv"
)

OUT_FULL = SRC.with_name(
    "NorESM2-LM_esm-hist_r1i1p1f1_1850-2014_AERA_OmegaA_full_input.csv"
)

OUT_MIN = SRC.with_name(
    "NorESM2-LM_esm-hist_r1i1p1f1_1850-2014_AERA_OmegaA_ff_input.csv"
)

y0 = 1850
y_hist_end = 2014

# Choose OA metric
OA_COL = "omega_arag_global_mean"
# Alternative if needed:
# OA_COL = "omega_calc_global_mean"


# ============================================================
# Read historical global annual time series
# ============================================================
hist = pd.read_csv(SRC)
hist["year"] = hist["year"].astype(int)
hist = hist.set_index("year").sort_index()

target_years = np.arange(y0, y_hist_end + 1)

missing_years = target_years[~np.isin(target_years, hist.index.values)]
if len(missing_years) > 0:
    raise ValueError(
        f"Missing historical years in source CSV: {missing_years[:10]}"
    )

required_cols = [OA_COL, "co2_surface_emission_PgC_yr"]
missing_cols = [c for c in required_cols if c not in hist.columns]
if missing_cols:
    raise ValueError(
        f"Source CSV is missing required columns: {missing_cols}"
    )

if hist.loc[target_years, OA_COL].isna().any():
    bad = hist.loc[target_years].index[hist.loc[target_years, OA_COL].isna()]
    raise ValueError(f"{OA_COL} has NaN values for years: {bad[:10].tolist()}")

if hist.loc[target_years, "co2_surface_emission_PgC_yr"].isna().any():
    bad = hist.loc[target_years].index[
        hist.loc[target_years, "co2_surface_emission_PgC_yr"].isna()
    ]
    raise ValueError(
        f"co2_surface_emission_PgC_yr has NaN values for years: {bad[:10].tolist()}"
    )


# ============================================================
# Minimal AERA-OA input
# ============================================================
df_min = pd.DataFrame(index=target_years)
df_min.index.name = "year"

# Store OA metric explicitly
df_min["omega_arag_global_mean"] = hist.loc[target_years, OA_COL].astype(float)

# Legacy-compatible alias:
# If the AERA/OA code still expects df["OmegaA"], this column is used as the control variable.
df_min["OmegaA"] = df_min["omega_arag_global_mean"].values

# Historical fossil-fuel CO2 emission
df_min["ff_emission"] = hist.loc[
    target_years, "co2_surface_emission_PgC_yr"
].astype(float)

df_min.to_csv(OUT_MIN)


# ============================================================
# Full AERA dataframe input
# ============================================================
df = aera_oa.get_base_df()

# Put OmegaA into df["OmegaA"] for compatibility with AERA core.
df.loc[target_years, "OmegaA"] = df_min["OmegaA"].values

# Also keep the real variable name as an extra column for clarity.
df["omega_arag_global_mean"] = np.nan
df.loc[target_years, "omega_arag_global_mean"] = df_min["omega_arag_global_mean"].values

# Fill fossil-fuel CO2 emissions
df.loc[target_years, "ff_emission"] = df_min["ff_emission"].values

# Save full dataframe.
# This keeps default lu_emission and non_co2_emission from AERA.
df.to_csv(OUT_FULL)


# ============================================================
# Check
# ============================================================
print(f"Wrote minimal OA input: {OUT_MIN}")
print(f"Wrote full OA input: {OUT_FULL}")

print("\nCheck: early years")
print(
    df.loc[
        1850:1855,
        ["OmegaA", "omega_arag_global_mean", "ff_emission", "lu_emission", "non_co2_emission"],
    ]
)

print("\nCheck: recent historical years")
print(
    df.loc[
        2010:2014,
        ["OmegaA", "omega_arag_global_mean", "ff_emission", "lu_emission", "non_co2_emission"],
    ]
)

print("\nNote:")
print("For this OA input, df['OmegaA'] is used as the AERA control-variable slot.")
print(f"df['OmegaA'] == df['omega_arag_global_mean'] from {y0} to {y_hist_end}.")

Use the following non-CO2 emission file: /nird/home/yongyub/kimyy/Dropbox/source/python/all/AERA/AERA_v2.0/AERA/aera_oa/data/nonco2_emis_ssp126_v3.dat
Use the following land use emission file: /nird/home/yongyub/kimyy/Dropbox/source/python/all/AERA/AERA_v2.0/AERA/aera_oa/data/lu_emis_ssp126_bern3d_adj_GCB2020_v1.dat
Use the following historical fossil fuel CO2 emission file: /nird/home/yongyub/kimyy/Dropbox/source/python/all/AERA/AERA_v2.0/AERA/aera_oa/data/co2_ff_GCP_plus_NDC_v1.dat
Wrote minimal OA input: /nird/datalake/NS2980K/users/yongyub/O2_linearlity/TipESM/cmor/esm-up2p0/v20251010/AERA/input/references/NorESM2-LM_esm-hist_r1i1p1f1_1850-2014_AERA_OmegaA_ff_input.csv
Wrote full OA input: /nird/datalake/NS2980K/users/yongyub/O2_linearlity/TipESM/cmor/esm-up2p0/v20251010/AERA/input/references/NorESM2-LM_esm-hist_r1i1p1f1_1850-2014_AERA_OmegaA_full_input.csv

Check: early years
        OmegaA  omega_arag_global_mean  ff_emission  lu_emission  \
year                                  

In [3]:
import numpy as np
import pandas as pd
from pathlib import Path

# ============================================================
# Use oxygen version of AERA
# ============================================================
import aera_oxygen as aera


# ============================================================
# Input / output
# ============================================================
SRC = Path(
    "/nird/datalake/NS2980K/users/yongyub/O2_linearlity/"
    "TipESM/cmor/esm-up2p0/v20251010/AERA/input/references/"
    "NorESM2-LM_esm-hist_r1i1p1f1_1850-2014_annual_global_timeseries.csv"
)

OUT_FULL = SRC.with_name(
    "NorESM2-LM_esm-hist_r1i1p1f1_1850-2014_AERA_O2var_0_200m_full_input.csv"
)

OUT_MIN = SRC.with_name(
    "NorESM2-LM_esm-hist_r1i1p1f1_1850-2014_AERA_O2var_0_200m_ff_input.csv"
)

y0 = 1850
y_hist_end = 2014

# Oxygen metric to use from the historical global annual timeseries
O2_SRC_COL = "o2_inventory_0_200m_Pmol"

# Column name expected by aera_oxygen
O2_AERA_COL = "O2var"


# ============================================================
# Read historical global annual time series
# ============================================================
hist = pd.read_csv(SRC)
hist["year"] = hist["year"].astype(int)
hist = hist.set_index("year").sort_index()

target_years = np.arange(y0, y_hist_end + 1)

missing_years = target_years[~np.isin(target_years, hist.index.values)]
if len(missing_years) > 0:
    raise ValueError(
        f"Missing historical years in source CSV: {missing_years[:10]}"
    )

# Historical source file uses co2_surface_emission_PgC_yr.
# If you later use an AERA-ready file, ff_emission may already exist.
if "ff_emission" in hist.columns:
    FF_COL = "ff_emission"
elif "co2_surface_emission_PgC_yr" in hist.columns:
    FF_COL = "co2_surface_emission_PgC_yr"
else:
    raise ValueError(
        "Source CSV must contain either 'ff_emission' or "
        "'co2_surface_emission_PgC_yr'."
    )

required_cols = [O2_SRC_COL, FF_COL]
missing_cols = [c for c in required_cols if c not in hist.columns]
if missing_cols:
    raise ValueError(
        f"Source CSV is missing required columns: {missing_cols}"
    )

if hist.loc[target_years, O2_SRC_COL].isna().any():
    bad = hist.loc[target_years].index[
        hist.loc[target_years, O2_SRC_COL].isna()
    ]
    raise ValueError(
        f"{O2_SRC_COL} has NaN values for years: {bad[:10].tolist()}"
    )

if hist.loc[target_years, FF_COL].isna().any():
    bad = hist.loc[target_years].index[
        hist.loc[target_years, FF_COL].isna()
    ]
    raise ValueError(
        f"{FF_COL} has NaN values for years: {bad[:10].tolist()}"
    )


# ============================================================
# Minimal AERA-Oxygen input
# ============================================================
df_min = pd.DataFrame(index=target_years)
df_min.index.name = "year"

# Store O2 inventory in the column expected by aera_oxygen.
# Unit: Pmol, because O2_SRC_COL is o2_inventory_0_200m_Pmol.
df_min[O2_AERA_COL] = hist.loc[target_years, O2_SRC_COL].astype(float)

# Keep the original source variable name as a reference column.
df_min[O2_SRC_COL] = hist.loc[target_years, O2_SRC_COL].astype(float)

# Historical fossil-fuel CO2 emission
df_min["ff_emission"] = hist.loc[target_years, FF_COL].astype(float)

df_min.to_csv(OUT_MIN)


# ============================================================
# Full AERA-Oxygen dataframe input
# ============================================================
df = aera.get_base_df()

# If the installed package still has another oxygen column from older code,
# this ensures O2var exists.
if O2_AERA_COL not in df.columns:
    df[O2_AERA_COL] = np.nan

# Fill O2var and fossil-fuel CO2 emissions
df.loc[target_years, O2_AERA_COL] = df_min[O2_AERA_COL].values
df.loc[target_years, "ff_emission"] = df_min["ff_emission"].values

# Keep original source variable name as an extra column for clarity.
df[O2_SRC_COL] = np.nan
df.loc[target_years, O2_SRC_COL] = df_min[O2_SRC_COL].values

# If the source CSV also contains lu_emission / non_co2_emission, use them.
# Otherwise, the default aera_oxygen values are kept.
for col in ["lu_emission", "non_co2_emission"]:
    if col in hist.columns:
        valid_years = hist.index[hist[col].notna()]
        valid_years = valid_years[
            (valid_years >= y0) & (valid_years <= df.index.max())
        ]
        df.loc[valid_years, col] = hist.loc[valid_years, col].values

# Optional: if your old installed get_base_df still creates "Oxygen",
# remove it to avoid confusion. Keep this commented if you want to inspect it.
if "Oxygen" in df.columns and O2_AERA_COL in df.columns:
    df = df.drop(columns=["Oxygen"])

# Save full dataframe.
df.to_csv(OUT_FULL)


# ============================================================
# Check
# ============================================================
print(f"Wrote minimal O2 input: {OUT_MIN}")
print(f"Wrote full O2 input: {OUT_FULL}")

print("\nCheck: early years")
print(
    df.loc[
        1850:1855,
        [O2_AERA_COL, O2_SRC_COL, "ff_emission", "lu_emission", "non_co2_emission"],
    ]
)

print("\nCheck: recent historical years")
print(
    df.loc[
        2010:2014,
        [O2_AERA_COL, O2_SRC_COL, "ff_emission", "lu_emission", "non_co2_emission"],
    ]
)

print("\nBasic diagnostics")
print(f"{O2_AERA_COL} unit: Pmol")
print(f"1850-1900 mean {O2_AERA_COL}: {df.loc[1850:1900, O2_AERA_COL].mean()}")
print(f"2014 {O2_AERA_COL}: {df.loc[2014, O2_AERA_COL]}")
print(f"2014 ff_emission: {df.loc[2014, 'ff_emission']}")

Use the following non-CO2 emission file: /nird/home/yongyub/kimyy/Dropbox/source/python/all/AERA/AERA_v2.1/AERA/aera_oxygen/data/nonco2_emis_ssp126_v3.dat
Use the following land use emission file: /nird/home/yongyub/kimyy/Dropbox/source/python/all/AERA/AERA_v2.1/AERA/aera_oxygen/data/lu_emis_ssp126_bern3d_adj_GCB2020_v1.dat
Use the following historical fossil fuel CO2 emission file: /nird/home/yongyub/kimyy/Dropbox/source/python/all/AERA/AERA_v2.1/AERA/aera_oxygen/data/co2_ff_GCP_plus_NDC_v1.dat
Wrote minimal O2 input: /nird/datalake/NS2980K/users/yongyub/O2_linearlity/TipESM/cmor/esm-up2p0/v20251010/AERA/input/references/NorESM2-LM_esm-hist_r1i1p1f1_1850-2014_AERA_O2var_0_200m_ff_input.csv
Wrote full O2 input: /nird/datalake/NS2980K/users/yongyub/O2_linearlity/TipESM/cmor/esm-up2p0/v20251010/AERA/input/references/NorESM2-LM_esm-hist_r1i1p1f1_1850-2014_AERA_O2var_0_200m_full_input.csv

Check: early years
          O2var  o2_inventory_0_200m_Pmol  ff_emission  lu_emission  \
year       